In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

In [ ]:
# knowledge base 

knowledge_base = [
    "To reset your password, click on the Forgot Password link on the login page.",
    "Billing invoices can be downloaded from the billing section of your account.",
    "If your account is locked, contact customer support for assistance.",
    "You can update your profile information from the account settings page.",
    "Subscription plans can be upgraded or downgraded anytime.",
    "If you cannot log in, ensure your username and password are correct.",
    "Refund requests are processed within five to seven business days.",
    "Two-factor authentication adds an extra layer of account security.",
    "Payment methods can be managed in the billing dashboard.",
    "Account deletion requests must be submitted through the support portal."
]

In [ ]:
# embedding model 

print("Loading embedding model...")
model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
# generate embeddings 

embeddings = model.encode(knowledge_base)

print("\nEmbedding Matrix Shape:")
print(embeddings.shape)

In [ ]:
# build faiss index 

dimension = embeddings.shape[1]  

index = faiss.IndexFlatL2(dimension)

# Normalize embeddings for cosine similarity behavior
embeddings = embeddings.astype("float32")
faiss.normalize_L2(embeddings)

# Add vectors to FAISS
index.add(embeddings)

print("\nTotal Vectors Stored in FAISS:")
print(index.ntotal)

In [6]:
# semantic search function 

def semantic_search(query, k=3):
    query_embedding = model.encode([query]).astype("float32")

    faiss.normalize_L2(query_embedding)

    distances, indices = index.search(query_embedding, k)

    print("\nTop Matches:")
    print("-" * 90)
    print(f"{'Rank':<6}{'Score':<15}{'Matched Sentence'}")
    print("-" * 90)

    for rank, (idx, score) in enumerate(
        zip(indices[0], distances[0]), start=1
    ):
        print(
            f"{rank:<6}{score:<15.4f}{knowledge_base[idx]}"
        )


In [ ]:
# task 3 test queries 

test_queries = [
    "How do I change my password?",
    "Where can I see my invoice?",
    "I am unable to access my account"
]


for query in test_queries:
    print(f"\nQuery: {query}")
    semantic_search(query)

In [ ]:
# task 4 interactive CLI 

while True:
    query = input("\nEnter your query (or type 'exit'): ")

    if query.lower() == "exit":
        print("Goodbye!")
        break

    semantic_search(query)